# Seaborn + Plotly

Seaborn helps with statistical summaries, while Plotly brings interactivity.


**Learning objectives**
- Create Seaborn box/violin and faceted plots
- Build a heatmap from a pivot table
- Use Plotly for interactive time series and maps

**Estimated time:** 60-75 minutes


## Setup


In [ ]:
from pathlib import Path
import sys

BASE_DIR = Path.cwd()
if (BASE_DIR / "viz-workshop").exists():
    BASE_DIR = BASE_DIR / "viz-workshop"
elif (BASE_DIR / "notebooks").exists():
    BASE_DIR = BASE_DIR.parent

sys.path.append(str(BASE_DIR / "../../src"))

from data_prep import load_or_create_dataset, get_paths, print_environment_hint

print_environment_hint()
paths = get_paths(BASE_DIR)

In [ ]:
# Load or build the processed dataset (cached after first run)
df = load_or_create_dataset(base_dir=BASE_DIR)
print(df.shape)
df.head()

## Seaborn Episode 1: Seaborn sits on Matplotlib


In [ ]:
import seaborn as sns
from viz_helpers import set_seaborn_theme

set_seaborn_theme()

sns.boxplot(data=df, x="day_name", y="duration_min", hue="member_casual")

## Seaborn Episode 2: Facet by rider type


In [ ]:
by_hour = df.groupby(["hour", "member_casual"]).size().reset_index(name="trips")

sns.catplot(
    data=by_hour,
    x="hour",
    y="trips",
    col="member_casual",
    kind="line",
    height=3,
    aspect=1.2,
)

## Seaborn Episode 3: Heatmap (day of week x hour)


In [ ]:
import pandas as pd
heat = df.groupby(["day_name", "hour"]).size().reset_index(name="trips")
heat_pivot = heat.pivot_table(index="day_name", columns="hour", values="trips")

sns.heatmap(heat_pivot, cmap="YlGnBu")

## Plotly Episode 1: Interactive time series


In [ ]:
import plotly.express as px

trips_by_day = df.groupby("date").size().reset_index(name="trips")
fig = px.line(trips_by_day, x="date", y="trips", title="Daily Citi Bike Trips")
fig.show()

## Plotly Episode 2: Interactive histogram


In [ ]:
fig = px.histogram(df, x="duration_min", nbins=50, title="Trip Duration")
fig.show()

## Plotly Episode 3: Station map (no Mapbox token)


In [ ]:
# Aggregate starts by station with lat/lng
stations = (
    df.dropna(subset=["start_lat", "start_lng", "start_station_name"])
      .groupby(["start_station_name", "start_lat", "start_lng"])
      .size()
      .reset_index(name="trips")
      .sort_values("trips", ascending=False)
      .head(200)
)

fig = px.scatter_mapbox(
    stations,
    lat="start_lat",
    lon="start_lng",
    size="trips",
    hover_name="start_station_name",
    zoom=10,
    mapbox_style="open-street-map",
    title="Top Start Stations (sample)"
)
fig.show()

## Exercises
1. Create a heatmap of usage by hour/day and interpret commuting patterns.
2. Build an interactive Plotly chart with a dropdown for membership type.

**Check yourself:** hover tooltips should show values and categories.


**Solution (optional): heatmap by hour and day**

In [ ]:
import pandas as pd
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
heat = df.groupby(["day_name", "hour"]).size().reset_index(name="trips")
heat["day_name"] = pd.Categorical(heat["day_name"], categories=weekday_order, ordered=True)
heat_pivot = heat.pivot_table(index="day_name", columns="hour", values="trips")

sns.heatmap(heat_pivot, cmap="YlGnBu")

## Common pitfalls
- Heatmaps: use a consistent day order to avoid confusing patterns.
- Maps: drop rows with missing lat/lng before mapping.


## Wrap-up
Seaborn is great for statistical summaries. Plotly shines when you want interactivity. Next, we combine everything in Dash.

**What’s next:** `04_dash_showcase.ipynb`
